In [2]:
import pandas as pd
import geopandas as gpd

In [ ]:
# Load the data
# Using low_memory=False because the listings file has many columns with mixed types
df_listings = pd.read_csv('raw_data/Airbnb Listings Data.csv.gz', low_memory=False)
df_calendar = pd.read_csv('raw_data/Airbnb Calendar Data.csv.gz')
df_reviews = pd.read_csv('raw_data/Airbnb Reviews.csv.gz')

# Load the geospatial data
gdf_neighbourhoods = gpd.read_file('raw_data/Neighbourhoods Data.geojson')

In [7]:
print(f"Listings shape: {df_listings.shape}")
print(f"Calendar shape: {df_calendar.shape}")
print(f"Reviews shape: {df_reviews.shape}")

Listings shape: (96871, 79)
Calendar shape: (35357974, 7)
Reviews shape: (2097996, 6)


In [8]:
# Calculate percentage of missing values per column in listings
missing_percentages = (df_listings.isnull().sum() / len(df_listings)) * 100
print(missing_percentages[missing_percentages > 0].sort_values(ascending=False))

neighbourhood_group_cleansed    100.000000
license                         100.000000
calendar_updated                100.000000
neighborhood_overview            57.460953
neighbourhood                    57.459921
host_neighbourhood               52.669013
host_about                       48.557360
beds                             36.047940
price                            36.035552
estimated_revenue_l365d          36.035552
bathrooms                        35.971550
host_response_time               32.731158
host_response_rate               32.731158
host_acceptance_rate             28.656667
review_scores_location           24.946578
review_scores_value              24.946578
review_scores_checkin            24.945546
review_scores_communication      24.921803
review_scores_accuracy           24.916642
review_scores_cleanliness        24.910448
reviews_per_month                24.901157
first_review                     24.901157
last_review                      24.901157
review_scor

In [12]:
# Calculate percentage of missing values per column in calendar
missing_percentages = (df_calendar.isnull().sum() / len(df_calendar)) * 100
print(missing_percentages[missing_percentages > 0].sort_values(ascending=False))

price             100.0
adjusted_price    100.0
dtype: float64


In [13]:
df_calendar[['price','adjusted_price']].isna().all()
df_calendar[['listing_id','date','available']].head()

,listing_id,date,available
0,1196288722069341420,2025-09-15,f
1,1196288722069341420,2025-09-16,f
2,1196288722069341420,2025-09-17,f
3,1196288722069341420,2025-09-18,f
4,1196288722069341420,2025-09-19,f


In [9]:
#formatting of the price column
print(df_listings[['id', 'price']].head())

      id    price
0  13913   $70.00
1  15400  $149.00
2  17402  $411.00
3  24328      NaN
4  36274  $210.00


In [14]:
#Drop the rows where price is NaN
df_listings_clean = df_listings.dropna(subset=['price']).copy()

# Strip the '$' and any commas, then cast to float
df_listings_clean['price_numeric'] = (
    df_listings_clean['price']
    .str.replace(r'[\$,]', '', regex=True)
    .astype(float)
)

#Verify the fix
print(df_listings_clean[['id', 'price', 'price_numeric']].head())

      id    price  price_numeric
0  13913   $70.00           70.0
1  15400  $149.00          149.0
2  17402  $411.00          411.0
4  36274  $210.00          210.0
5  36299  $280.00          280.0


In [10]:
print(f"Calendar starts: {df_calendar['date'].min()} and ends: {df_calendar['date'].max()}")
print(df_calendar['available'].value_counts())

Calendar starts: 2025-09-14 and ends: 2026-09-17
available
f    21313866
t    14044108
Name: count, dtype: int64


In [11]:
sample_id = df_listings['id'].iloc[0]
print(f"Listing ID: {sample_id}")
print(f"Calendar entries for this ID: {len(df_calendar[df_calendar['listing_id'] == sample_id])}")
print(f"Reviews for this ID: {len(df_reviews[df_reviews['listing_id'] == sample_id])}")

Listing ID: 13913
Calendar entries for this ID: 365
Reviews for this ID: 55
